In [ ]:
import mne
from mne.decoding import CSP
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import os
from mne import Epochs, events_from_annotations, pick_types
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
import joblib
import pandas as pd
import glob
from mrmr import mrmr_classif


le = LabelEncoder()
random_state=42
mne.set_log_level('CRITICAL')

In [ ]:
colors = {
    'AW': '#ef8a62',
    'MO': '#67a9cf',
    'MI': '#999999',
    'AW_2': '#613828'
}
data_folder = "raw_data"
#data_folder = "cleaned_dataset"
ERP_channels =  ['Fz', 'C3', 'Cz', 'C4', 'Pz', 'Oz'] # Updated channel list
FBCSP_channels = ['C3', 'Cz', 'C4', 'PO7', 'Pz', 'PO8'] # Updated channel list

In [13]:
def create_offline_epochs(raw, events, event_id, channels, tmin_tmax, baseline, clean_data=True):
    
    if clean_data:
        raw.filter(l_freq=1.0, h_freq=None,  verbose=False)
        raw.filter(l_freq=None, h_freq=40.0,  verbose=False)
    picks = pick_types(raw.info, meg=False, eeg=True, stim=False, eog=False)
    epochs = Epochs(
            raw,
            events,
            event_id,
            tmin=tmin_tmax[0],
            tmax=tmin_tmax[1],
            proj=False,
            picks=picks,
            baseline=baseline,
            preload=True,
         verbose=False)
    
    return epochs.pick_channels(channels) 

In [5]:
def select_discriminative_band(X, y, channels, sfreq, f_range=(5, 35), threshold=0.05):
    from scipy.signal import welch
    from scipy.stats import pearsonr
    """
    Selects a discriminative frequency band from EEG data.

    Args:
        X (numpy.ndarray): EEG data of shape (n_trials, n_channels, n_samples).
        y (numpy.ndarray): Trial labels of shape (n_trials,).
        channels (list): List of channel names.
        sfreq (int): Sampling frequency of the EEG data.
        f_range (tuple): Frequency range to consider (default: 5-35 Hz).
        threshold (float): Threshold for band expansion (default: 0.05).

    Returns:
        tuple: Selected frequency band [f0, f1].
    """
    n_trials, n_channels, n_samples = X.shape
    f_start, f_end = f_range
    
    scores = {}
    for f in range(f_start, f_end + 1):
        scores[f] = np.zeros(n_channels)
        for c_idx, c in enumerate(channels):
            band_power = []
            for i in range(n_trials):
                # Compute power spectral density using Welch's method
                f_welch, psd = welch(X[i, c_idx, :], sfreq, window='hamming', nfft=n_samples)
                # Find the power at frequency f
                f_idx = np.argmin(np.abs(f_welch - f))
                band_power.append(np.log(psd[f_idx]))
            
            # Calculate correlation between band power and labels
            correlation, _ = pearsonr(band_power, y)
            scores[f][c_idx] = correlation if not np.isnan(correlation) else 0.0  # Handle NaN

    # Find the channel with the maximum correlation for each frequency
    f_max_channels = {f: channels[np.argmax(np.abs(scores[f]))] for f in scores}

    # Find the frequency with the maximum absolute correlation across all channels
    f_max = max(scores, key=lambda f: np.max(np.abs(scores[f])))
    f_score_max = np.max(np.abs(scores[f_max]))

    # Initialize frequency band
    f0 = f_max
    f1 = f_max

    # Expand lower bound
    while f0 - 1 >= f_start and np.max(np.abs(scores[f0 - 1])) >= f_score_max * (1 - threshold):
        f0 -= 1

    # Expand upper bound
    while f1 + 1 <= f_end and np.max(np.abs(scores[f1 + 1])) >= f_score_max * (1 - threshold):
        f1 += 1

    return f0-3, f1+3

In [6]:
def perform_mrmr_feature_selection(features, labels, num_features=10):
    # Convert the scaled features to a pandas DataFrame
    df = pd.DataFrame(features)

    # Add target variable to the DataFrame
    df['target'] = labels

    # Convert target to integer type (required by mRMR)
    df['target'] = df['target'].astype(int)

    # Perform MRMR feature selection
    # Note: The features argument here is already scaled by the calling function
    selected_features = mrmr_classif(df.drop(columns='target'), df['target'], num_features, show_progress=False)

    # Create datasets with selected features
    X_selected = features[:, selected_features]
    return X_selected, selected_features


In [7]:
def extract_fbcsp_features_for_cv(epochs, filtered_bands=None):
    if filtered_bands is None:
        filtered_bands = [(8, 12), (12, 16), (16, 20), (20, 24), (24, 28), (13, 30)]
    filtered_epochs_list = []
    for l_freq, h_freq in filtered_bands:
        epochs_filtered = epochs.copy().filter(l_freq=l_freq, h_freq=h_freq,
                                               fir_design='firwin', verbose=False)
        filtered_epochs_list.append(epochs_filtered)
    return filtered_epochs_list

In [ ]:
def extract_fbcsp_features_and_model(epochs, labels, random_state=42):
    clf = SVC(kernel='linear', random_state=random_state, C=0.5)

    #clf = RandomForestClassifier(n_estimators=100, random_state=random_state, n_jobs=-1)
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=random_state)
    accuracies = []

    filtered_epochs_list = extract_fbcsp_features_for_cv(epochs)

    for train_index, test_index in skf.split(epochs, labels):
        csp_features_list_train = []
        csp_features_list_test = []
        n_components = 4
        csp = CSP(n_components=n_components, reg='ledoit_wolf', log=True, norm_trace=False)
        y_train = labels[train_index]
        y_test = labels[test_index]

        for epochs_filtered in filtered_epochs_list:
            epochs_data = epochs_filtered.get_data(copy=False) * 1e6
            X_train_fold = epochs_data[train_index]
            X_test_fold = epochs_data[test_index]

            csp.fit(X_train_fold, y_train)
            csp_features_train = csp.transform(X_train_fold)
            csp_features_test = csp.transform(X_test_fold)

            csp_features_list_train.append(csp_features_train)
            csp_features_list_test.append(csp_features_test)

        fbcsp_features_train = np.concatenate(csp_features_list_train, axis=1)
        fbcsp_features_test = np.concatenate(csp_features_list_test, axis=1)
        
        # Instantiate and fit scaler for this fold
        scaler = StandardScaler()
        fbcsp_features_train_scaled = scaler.fit_transform(fbcsp_features_train)
        fbcsp_features_test_scaled = scaler.transform(fbcsp_features_test)
        
        mrmr_fbcsp_features_train, selected_features = perform_mrmr_feature_selection(fbcsp_features_train_scaled, y_train, num_features=10)
        mrmr_fbcsp_features_test = fbcsp_features_test_scaled[:, selected_features]

        clf.fit(mrmr_fbcsp_features_train, y_train)
        y_pred = clf.predict(mrmr_fbcsp_features_test)
        accuracy = accuracy_score(y_test, y_pred)
        accuracies.append(accuracy)

    final_csp_features_list = []
    final_csp = CSP(n_components=4, reg='ledoit_wolf', log=True, norm_trace=False)
    for epochs_filtered in filtered_epochs_list:
        epochs_data = epochs_filtered.get_data(copy=False) * 1e6
        final_csp.fit(epochs_data, labels)
        final_csp_features_list.append(final_csp.transform(epochs_data))
    final_fbcsp_features = np.concatenate(final_csp_features_list, axis=1)

    final_scaler = StandardScaler()
    final_fbcsp_features_scaled = final_scaler.fit_transform(final_fbcsp_features)

    mrmr_final_fbcsp_features, selected_features = perform_mrmr_feature_selection(final_fbcsp_features_scaled, labels, num_features=10)
    
    final_clf = SVC(kernel='linear', random_state=random_state, C=0.5)
    final_clf.fit(mrmr_final_fbcsp_features, labels)
    
    return {'average_accuracy': np.mean(accuracies), 'selected_features': selected_features}, final_clf, final_csp, final_scaler, mrmr_final_fbcsp_features

In [ ]:
def offline_combined_classification(fbcsp_features, erp_features, labels):
    """
    Performs classification on a combined feature set (FBCSP and ERP).
    """
    clf = SVC(kernel='linear', random_state=random_state, C=0.5)

    #clf = RandomForestClassifier(n_estimators=100, random_state=random_state, n_jobs=-1)
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=random_state)
    accuracies = []

    combined_features = np.concatenate([fbcsp_features, erp_features], axis=1)
    
    for train_index, test_index in skf.split(combined_features, labels):
        X_train, X_test = combined_features[train_index], combined_features[test_index]
        y_train, y_test = labels[train_index], labels[test_index]
        
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        accuracies.append(accuracy)
    
    final_clf = SVC(kernel='linear', random_state=random_state, C=0.5)

    #final_clf = RandomForestClassifier(n_estimators=100, random_state=random_state, n_jobs=-1)
    final_clf.fit(combined_features, labels)
    
    return {'average_accuracy': np.mean(accuracies)}, final_clf, combined_features


In [ ]:
def extract_erp_features_and_model(epochs, labels, random_state=42):
    clf = SVC(kernel='linear', random_state=random_state, C=0.5)
    #clf = RandomForestClassifier(n_estimators=100, random_state=random_state, n_jobs=-1)
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=random_state)
    accuracies = []

    window_duration_s = 0.5
    step_size_s = 0.5
    sfreq = epochs.info['sfreq']
    window_samples = int(window_duration_s * sfreq)
    step_samples = int(step_size_s * sfreq)
    start_time_s = 0.0
    end_time_s = epochs.tmax
    start_time_sample = epochs.time_as_index(start_time_s)[0]
    end_time_sample = epochs.time_as_index(end_time_s)[0]

    all_erp_features = []
    for epoch_data in epochs.get_data():
        trial_features = []
        for start_idx in np.arange(start_time_sample, end_time_sample - window_samples + 1, step_samples):
            end_idx = start_idx + window_samples
            window_data = epoch_data[:, start_idx:end_idx]
            window_features = np.mean(window_data, axis=1)
            trial_features.append(window_features)
        all_erp_features.append(np.concatenate(trial_features))
    final_erp_features = np.array(all_erp_features)

    for train_index, test_index in skf.split(final_erp_features, labels):
        X_train, X_test = final_erp_features[train_index], final_erp_features[test_index]
        y_train, y_test = labels[train_index], labels[test_index]

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        mrmr_train_features, selected_features = perform_mrmr_feature_selection(X_train_scaled, y_train, num_features=10)
        mrmr_test_features = X_test_scaled[:, selected_features]

        clf.fit(mrmr_train_features, y_train)
        y_pred = clf.predict(mrmr_test_features)
        accuracy = accuracy_score(y_test, y_pred)
        accuracies.append(accuracy)

    final_scaler = StandardScaler()
    final_erp_features_scaled = final_scaler.fit_transform(final_erp_features)
    mrmr_final_erp_features, selected_features = perform_mrmr_feature_selection(final_erp_features_scaled, labels, num_features=10)
    final_erp_features_selected = final_erp_features_scaled[:, selected_features]

    #final_clf = RandomForestClassifier(n_estimators=100, random_state=random_state, n_jobs=-1)
    final_clf = SVC(kernel='linear', random_state=random_state, C=0.5)

    final_clf.fit(final_erp_features_selected, labels)

    return {'average_accuracy': np.mean(accuracies), 'selected_features': selected_features}, final_clf, final_scaler, final_erp_features_selected

In [11]:
def align_epochs_and_labels(fbcsp_epochs, erp_epochs, arm_label='Start_cue_Arm', leg_label='Start_cue_Leg'):

    fbcsp_events = fbcsp_epochs.events
    erp_events = erp_epochs.events

            # Find common events
    common_event_ids_fbcsp = set(tuple(row) for row in fbcsp_events)
    common_event_ids_erp = set(tuple(row) for row in erp_events)
    common_events = list(common_event_ids_fbcsp.intersection(common_event_ids_erp))

            # Get indices for alignment
    fbcsp_indices = [np.where((fbcsp_events == common_event).all(axis=1))[0][0] for common_event in common_events]
    erp_indices = [np.where((erp_events == common_event).all(axis=1))[0][0] for common_event in common_events]

            # Align epochs
    fbcsp_aligned = fbcsp_epochs[fbcsp_indices]
    erp_aligned = erp_epochs[erp_indices]

            # Create label mapping and labels
    mapping_offline = {
    fbcsp_aligned[arm_label].events[:, -1][0]: 0,
    fbcsp_aligned[leg_label].events[:, -1][0]: 1}
    labels_offline = np.vectorize(mapping_offline.get)(fbcsp_aligned.events[:, -1])
    return fbcsp_aligned, erp_aligned, labels_offline

In [14]:
epochs_AW_FBCSP=[]
epochs_MI_FBCSP=[]
epochs_MO_FBCSP=[]
epochs_AW_ERP=[]
epochs_MI_ERP=[]
epochs_MO_ERP =[]
subjects = [str(i) for i in range(1,30)]

for subject in subjects:
    set_files = glob.glob(os.path.join(data_folder, subject, '*.set'))
    for file in set_files:
        filename = file
        raw = mne.io.read_raw_eeglab(filename, preload=True, verbose=False)  # Load data
        event_id = {'Start_cue_Arm': 3, 'Start_cue_Leg': 4}
        events, _ = events_from_annotations(raw, event_id, verbose=False)
        if 'AW' in file:
            epochs_AW_FBCSP.append(create_offline_epochs(raw, events, event_id, FBCSP_channels, [0.5, 3.0], None))#, clean_data=False))
            epochs_AW_ERP.append(create_offline_epochs(raw, events, event_id, ERP_channels, [-0.5, 2.5], [-0.5, 0]))#, clean_data=False))
        elif 'MI' in file:
            epochs_MI_FBCSP.append(create_offline_epochs(raw, events, event_id, FBCSP_channels, [0.5, 3.0], None))#, clean_data=False))
            epochs_MI_ERP.append(create_offline_epochs(raw, events, event_id, ERP_channels, [-0.5, 2.5], [-0.5, 0]))#, clean_data=False))
        elif 'MO' in file:
            epochs_MO_FBCSP.append(create_offline_epochs(raw, events, event_id, FBCSP_channels, [0.5, 3.0], None))##, clean_data=False))
            epochs_MO_ERP.append(create_offline_epochs(raw, events, event_id, ERP_channels, [-0.5, 2.5], [-0.5, 0]))##, clean_data=False))

In [15]:

AW_results = {'FBCSP': {}, 'ERP': {}, 'Combined': {}}
    
for n, subject in enumerate(subjects):
    AW_FBCSP = epochs_AW_FBCSP[n]
    AW_ERP = epochs_AW_ERP[n]
        
    AW_FBCSP_aligned, AW_ERP_aligned, labels_offline = align_epochs_and_labels(AW_FBCSP, AW_ERP)
        
    print(f"--- Processing Subject {subject} ---")

        # 1. FBCSP-only Classification
    fbcsp_results, fbcsp_clf, fbcsp_csp, fbcsp_scaler, fbcsp_features = extract_fbcsp_features_and_model(AW_FBCSP_aligned, labels_offline)
    AW_results['FBCSP'][subject] = fbcsp_results
    joblib.dump(fbcsp_clf, os.path.join('classifiers', f'AW_FBCSP_clf_subject_{n}.pkl'))
    joblib.dump(fbcsp_csp, os.path.join('features_extractors', f'AW_FBCSP_csp_subject_{n}.pkl'))
    joblib.dump(fbcsp_scaler, os.path.join('features_extractors', f'AW_FBCSP_csp_scaler_subject_{n}.pkl'))

    print(f"FBCSP-only accuracy: {fbcsp_results['average_accuracy']:.4f}")

    # 2. ERP-only Classification
    erp_results, erp_clf, erp_scaler, erp_features = extract_erp_features_and_model(AW_ERP_aligned, labels_offline)
    AW_results['ERP'][subject] = erp_results
    joblib.dump(erp_clf, os.path.join('classifiers', f'AW_ERP_clf_subject_{n}.pkl'))
    joblib.dump(erp_scaler, os.path.join('features_extractors', f'AW_ERP_scaler_subject_{n}.pkl'))

    print(f"ERP-only accuracy: {erp_results['average_accuracy']:.4f}")

        # 3. FBCSP + ERP Combined Classification
    combined_results, combined_clf, combined_features = offline_combined_classification(fbcsp_features, erp_features, labels_offline)
    AW_results['Combined'][subject] = combined_results
    joblib.dump(combined_clf, os.path.join('classifiers', f'AW_Combined_clf_subject_{n}.pkl'))
    print(f"Combined accuracy: {combined_results['average_accuracy']:.4f}")



--- Processing Subject 1 ---
FBCSP-only accuracy: 0.4125
ERP-only accuracy: 0.5250
Combined accuracy: 0.6000
--- Processing Subject 2 ---
FBCSP-only accuracy: 0.5125
ERP-only accuracy: 0.4500
Combined accuracy: 0.4500
--- Processing Subject 3 ---
FBCSP-only accuracy: 0.3750
ERP-only accuracy: 0.5500
Combined accuracy: 0.6000
--- Processing Subject 4 ---
FBCSP-only accuracy: 0.5000
ERP-only accuracy: 0.4625
Combined accuracy: 0.6250
--- Processing Subject 5 ---
FBCSP-only accuracy: 0.5375
ERP-only accuracy: 0.5375
Combined accuracy: 0.4750
--- Processing Subject 6 ---
FBCSP-only accuracy: 0.5000
ERP-only accuracy: 0.5750
Combined accuracy: 0.6875
--- Processing Subject 7 ---
FBCSP-only accuracy: 0.5375
ERP-only accuracy: 0.4625
Combined accuracy: 0.6625
--- Processing Subject 8 ---
FBCSP-only accuracy: 0.5500
ERP-only accuracy: 0.5250
Combined accuracy: 0.6000
--- Processing Subject 9 ---
FBCSP-only accuracy: 0.4500
ERP-only accuracy: 0.4250
Combined accuracy: 0.5000
--- Processing Subj

In [16]:
MI_results = {'FBCSP': {}, 'ERP': {}, 'Combined': {}}
    
for n, subject in enumerate(subjects):
    MI_FBCSP = epochs_MI_FBCSP[n]
    MI_ERP = epochs_MI_ERP[n]
        
    MI_FBCSP_aligned, MI_ERP_aligned, labels_offline = align_epochs_and_labels(MI_FBCSP, MI_ERP)
        
    print(f"--- Processing Subject {subject} ---")

    # 1. FBCSP-only Classification
    fbcsp_results, fbcsp_clf, fbcsp_csp, fbcsp_scaler, fbcsp_features = extract_fbcsp_features_and_model(MI_FBCSP_aligned, labels_offline)
    MI_results['FBCSP'][subject] = fbcsp_results
    joblib.dump(fbcsp_clf, os.path.join('classifiers', f'MI_FBCSP_clf_subject_{n}.pkl'))
    joblib.dump(fbcsp_csp, os.path.join('features_extractors', f'MI_FBCSP_csp_subject_{n}.pkl'))
    joblib.dump(fbcsp_scaler, os.path.join('features_extractors', f'MI_FBCSP_csp_scaler_subject_{n}.pkl'))

    print(f"FBCSP-only accuracy: {fbcsp_results['average_accuracy']:.4f}")

    # 2. ERP-only Classification
    erp_results, erp_clf, erp_scaler, erp_features = extract_erp_features_and_model(MI_ERP_aligned, labels_offline)
    MI_results['ERP'][subject] = erp_results
    joblib.dump(erp_clf, os.path.join('classifiers', f'MI_ERP_clf_subject_{n}.pkl'))
    joblib.dump(erp_scaler, os.path.join('features_extractors', f'MI_ERP_scaler_subject_{n}.pkl'))

    print(f"ERP-only accuracy: {erp_results['average_accuracy']:.4f}")

    # 3. FBCSP + ERP Combined Classification
    combined_results, combined_clf, combined_features = offline_combined_classification(fbcsp_features, erp_features, labels_offline)
    MI_results['Combined'][subject] = combined_results
    joblib.dump(combined_clf, os.path.join('classifiers', f'MI_Combined_clf_subject_{n}.pkl'))
    print(f"Combined accuracy: {combined_results['average_accuracy']:.4f}")


--- Processing Subject 1 ---
FBCSP-only accuracy: 0.6625
ERP-only accuracy: 0.5500
Combined accuracy: 0.6500
--- Processing Subject 2 ---
FBCSP-only accuracy: 0.5625
ERP-only accuracy: 0.5500
Combined accuracy: 0.6875
--- Processing Subject 3 ---
FBCSP-only accuracy: 0.4875
ERP-only accuracy: 0.6125
Combined accuracy: 0.7000
--- Processing Subject 4 ---
FBCSP-only accuracy: 0.5375
ERP-only accuracy: 0.5000
Combined accuracy: 0.7000
--- Processing Subject 5 ---
FBCSP-only accuracy: 0.4125
ERP-only accuracy: 0.4375
Combined accuracy: 0.5250
--- Processing Subject 6 ---
FBCSP-only accuracy: 0.4875
ERP-only accuracy: 0.6000
Combined accuracy: 0.6125
--- Processing Subject 7 ---
FBCSP-only accuracy: 0.5875
ERP-only accuracy: 0.5125
Combined accuracy: 0.6750
--- Processing Subject 8 ---
FBCSP-only accuracy: 0.5875
ERP-only accuracy: 0.4375
Combined accuracy: 0.5625
--- Processing Subject 9 ---
FBCSP-only accuracy: 0.5625
ERP-only accuracy: 0.5375
Combined accuracy: 0.7000
--- Processing Subj

In [17]:
MO_results = {'FBCSP': {}, 'ERP': {}, 'Combined': {}}
    
for n, subject in enumerate(subjects):
    MO_FBCSP = epochs_MO_FBCSP[n]
    MO_ERP = epochs_MO_ERP[n]
        
    MO_FBCSP_aligned, MO_ERP_aligned, labels_offline = align_epochs_and_labels(MO_FBCSP, MO_ERP)
        
    print(f"--- Processing Subject {subject} ---")

    # 1. FBCSP-only Classification
    fbcsp_results, fbcsp_clf, fbcsp_csp, fbcsp_scaler, fbcsp_features = extract_fbcsp_features_and_model(MO_FBCSP_aligned, labels_offline)
    MO_results['FBCSP'][subject] = fbcsp_results
    joblib.dump(fbcsp_clf, os.path.join('classifiers', f'MO_FBCSP_clf_subject_{n}.pkl'))
    joblib.dump(fbcsp_csp, os.path.join('features_extractors', f'MO_FBCSP_csp_subject_{n}.pkl'))
    joblib.dump(fbcsp_scaler, os.path.join('features_extractors', f'MO_FBCSP_csp_scaler_subject_{n}.pkl'))

    print(f"FBCSP-only accuracy: {fbcsp_results['average_accuracy']:.4f}")

    # 2. ERP-only Classification
    erp_results, erp_clf, erp_scaler, erp_features = extract_erp_features_and_model(MO_ERP_aligned, labels_offline)
    MO_results['ERP'][subject] = erp_results
    joblib.dump(erp_clf, os.path.join('classifiers', f'MO_ERP_clf_subject_{n}.pkl'))
    joblib.dump(erp_scaler, os.path.join('features_extractors', f'MO_ERP_scaler_subject_{n}.pkl'))

    print(f"ERP-only accuracy: {erp_results['average_accuracy']:.4f}")

    # 3. FBCSP + ERP Combined Classification
    combined_results, combined_clf, combined_features = offline_combined_classification(fbcsp_features, erp_features, labels_offline)
    MO_results['Combined'][subject] = combined_results
    joblib.dump(combined_clf, os.path.join('classifiers', f'MO_Combined_clf_subject_{n}.pkl'))
    print(f"Combined accuracy: {combined_results['average_accuracy']:.4f}")


--- Processing Subject 1 ---
FBCSP-only accuracy: 0.5625
ERP-only accuracy: 0.4875
Combined accuracy: 0.7125
--- Processing Subject 2 ---
FBCSP-only accuracy: 0.6375
ERP-only accuracy: 0.4750
Combined accuracy: 0.7000
--- Processing Subject 3 ---
FBCSP-only accuracy: 0.5125
ERP-only accuracy: 0.5000
Combined accuracy: 0.6250
--- Processing Subject 4 ---
FBCSP-only accuracy: 0.5125
ERP-only accuracy: 0.5125
Combined accuracy: 0.6750
--- Processing Subject 5 ---
FBCSP-only accuracy: 0.5625
ERP-only accuracy: 0.5000
Combined accuracy: 0.5375
--- Processing Subject 6 ---
FBCSP-only accuracy: 0.4625
ERP-only accuracy: 0.4875
Combined accuracy: 0.7125
--- Processing Subject 7 ---
FBCSP-only accuracy: 0.5375
ERP-only accuracy: 0.6500
Combined accuracy: 0.6375
--- Processing Subject 8 ---
FBCSP-only accuracy: 0.5625
ERP-only accuracy: 0.6000
Combined accuracy: 0.6125
--- Processing Subject 9 ---
FBCSP-only accuracy: 0.4125
ERP-only accuracy: 0.5125
Combined accuracy: 0.5375
--- Processing Subj

In [18]:
for subject in subjects:
    print(subject, AW_results['ERP'][subject]['average_accuracy'], MI_results['ERP'][subject]['average_accuracy'], MO_results['ERP'][subject]['average_accuracy'])


1 0.525 0.55 0.4875
2 0.45 0.55 0.475
3 0.55 0.6125 0.5
4 0.4625 0.5 0.5125
5 0.5375 0.4375 0.5
6 0.575 0.6 0.4875
7 0.4625 0.5125 0.65
8 0.525 0.4375 0.6
9 0.425 0.5375 0.5125
10 0.475 0.525 0.4625
11 0.5125 0.4625 0.4625
12 0.35 0.55 0.475
13 0.4375 0.5625 0.4875
14 0.65 0.5125 0.45
15 0.375 0.5 0.525
16 0.475 0.5125 0.525
17 0.525 0.525 0.4375
18 0.475 0.4125 0.45
19 0.5 0.5125 0.5125
20 0.525 0.6 0.55
21 0.55 0.5125 0.625
22 0.475 0.5375 0.4625
23 0.4875 0.625 0.5
24 0.6 0.3875 0.45
25 0.425 0.4875 0.5375
26 0.5875 0.6 0.475
27 0.5125 0.5 0.5125
28 0.4625 0.5625 0.4625
29 0.525 0.4875 0.525


In [19]:
for subject in subjects:
    print(subject, AW_results['FBCSP'][subject]['average_accuracy'], MI_results['FBCSP'][subject]['average_accuracy'], MO_results['FBCSP'][subject]['average_accuracy'])

1 0.4125 0.6625 0.5625
2 0.5125 0.5625 0.6375
3 0.375 0.4875 0.5125
4 0.5 0.5375 0.5125
5 0.5375 0.4125 0.5625
6 0.5 0.4875 0.4625
7 0.5375 0.5875 0.5375
8 0.55 0.5875 0.5625
9 0.45 0.5625 0.4125
10 0.4625 0.5 0.475
11 0.4375 0.525 0.6
12 0.625 0.625 0.425
13 0.55 0.5375 0.45
14 0.6 0.5625 0.4625
15 0.55 0.625 0.4625
16 0.5 0.475 0.4625
17 0.45 0.55 0.5625
18 0.4375 0.4625 0.5
19 0.5625 0.5125 0.4625
20 0.375 0.6125 0.3125
21 0.4625 0.45 0.525
22 0.4125 0.475 0.4375
23 0.525 0.5625 0.6375
24 0.6 0.5375 0.5125
25 0.4875 0.475 0.4875
26 0.5625 0.5375 0.475
27 0.5875 0.5375 0.6125
28 0.575 0.625 0.5125
29 0.4125 0.6 0.575


In [20]:
for subject in subjects:
    print(subject, AW_results['Combined'][subject]['average_accuracy'], MI_results['Combined'][subject]['average_accuracy'], MO_results['Combined'][subject]['average_accuracy'])

1 0.6 0.65 0.7125
2 0.45 0.6875 0.7
3 0.6 0.7 0.625
4 0.625 0.7 0.675
5 0.475 0.525 0.5375
6 0.6875 0.6125 0.7125
7 0.6625 0.675 0.6375
8 0.6 0.5625 0.6125
9 0.5 0.7 0.5375
10 0.525 0.625 0.5375
11 0.45 0.625 0.7
12 0.575 0.5375 0.6
13 0.55 0.7125 0.575
14 0.6 0.5625 0.425
15 0.4625 0.6 0.5
16 0.625 0.3375 0.6375
17 0.5 0.4875 0.6125
18 0.6625 0.5625 0.4375
19 0.6625 0.5 0.6875
20 0.6875 0.65 0.475
21 0.5625 0.525 0.6375
22 0.5125 0.6125 0.6625
23 0.575 0.625 0.6375
24 0.5875 0.4875 0.7375
25 0.525 0.7 0.5375
26 0.7125 0.6875 0.5875
27 0.6375 0.6875 0.75
28 0.6375 0.5375 0.4875
29 0.6625 0.6875 0.575


In [26]:
data = []
for subject_id, scores in AW_results['FBCSP'].items():
    data.append({
        'Subject': subject_id,
        'Condition': 'AW',
        'Feature_Extraction': 'FBCSP',
        'Accuracy': scores['average_accuracy'],
        'Selected_Features': scores['selected_features']
        })
for subject_id, scores in MO_results['FBCSP'].items():
    data.append({
        'Subject': subject_id,
        'Condition': 'MO',
        'Feature_Extraction': 'FBCSP',
        'Accuracy': scores['average_accuracy'],
        'Selected_Features': scores['selected_features']
        })
for subject_id, scores in MI_results['FBCSP'].items():
    data.append({
        'Subject': subject_id,
        'Condition': 'MI',
        'Feature_Extraction': 'FBCSP',        
        'Accuracy': scores['average_accuracy'],
        'Selected_Features': scores['selected_features']
        })
for subject_id, scores in AW_results['ERP'].items():
    data.append({
        'Subject': subject_id,
        'Condition': 'AW',
        'Feature_Extraction': 'ERP',
        'Accuracy': scores['average_accuracy'],
        'Selected_Features': scores['selected_features']
    })
for subject_id, scores in MO_results['ERP'].items():
    data.append({
        'Subject': subject_id,
        'Condition': 'MO',
        'Feature_Extraction': 'ERP',
        'Accuracy': scores['average_accuracy'],
        'Selected_Features': scores['selected_features']    })
for subject_id, scores in MI_results['ERP'].items():
    data.append({
        'Subject': subject_id,
        'Condition': 'MI',
        'Feature_Extraction': 'ERP',
        'Accuracy': scores['average_accuracy'],
        'Selected_Features': scores['selected_features']    })
for subject_id, scores in AW_results['Combined'].items():
    data.append({
        'Subject': subject_id,
        'Condition': 'AW',
        'Feature_Extraction': 'Combined',
        'Accuracy': scores['average_accuracy'],
        'Selected_Features': []
    })
for subject_id, scores in MO_results['Combined'].items():
    data.append({
        'Subject': subject_id,
        'Condition': 'MO',
        'Feature_Extraction': 'Combined',
        'Accuracy': scores['average_accuracy'],
        'Selected_Features': []
    })
for subject_id, scores in MI_results['Combined'].items():
    data.append({
        'Subject': subject_id,
        'Condition': 'MI',
        'Feature_Extraction': 'Combined',
        'Accuracy': scores['average_accuracy'],
        'Selected_Features': []
    })

df = pd.DataFrame(data)
df.to_excel('classification_results/offline_classification_results.xlsx', index=False)
df.drop(columns='Selected_Features').groupby(['Condition', 'Feature_Extraction'])['Accuracy'].agg(['mean', 'std'])

mean       std
Condition Feature_Extraction                    
AW        Combined            0.583190  0.076348
          ERP                 0.497845  0.064958
          FBCSP               0.501724  0.070134
MI        Combined            0.605603  0.089179
          ERP                 0.521121  0.058354
          FBCSP               0.540517  0.060748
MO        Combined            0.605172  0.089050
          ERP                 0.503879  0.051224
          FBCSP               0.507328  0.072235